# Task 3 — Professional Data Cleaning Project

**Objective:** Transform a deliberately messy employee dataset into a reliable, analysis-ready dataset while documenting every cleaning decision.

**Dataset:** Kaggle's *Messy Employee Dataset* (`Messy_Employee_dataset.csv`)—a synthetic HR dataset designed with missing values, inconsistent formats, invalid entries, compound fields, duplicates, incorrect data types, and outliers.

**Tech stack:** Python, pandas, NumPy, matplotlib, seaborn, Jupyter Notebook

---

### Deliverables

1. Initial data-quality report by column
2. Explicit missing-data strategy and justification
3. Duplicate identification and removal log
4. Text, category, ID, phone, email, and date standardisation
5. IQR-based numeric outlier analysis and treatment
6. Correct pandas data types
7. Before-versus-after quality summary
8. Validation checks for the cleaned data
9. Exported `Messy_Employee_dataset_cleaned.csv`

> The notebook uses a fixed reference date so future-date validation remains reproducible.

## 1. Environment setup

Run the installation cell once if your Jupyter environment does not already contain these packages.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn

In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from pandas.api.types import (
    is_bool_dtype,
    is_categorical_dtype,
    is_datetime64_any_dtype,
    is_float_dtype,
    is_integer_dtype,
)

warnings.filterwarnings("ignore")

REFERENCE_DATE = pd.Timestamp("2026-08-21")
OUTPUT_PATH = Path("Messy_Employee_dataset_cleaned.csv")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Environment ready.")

## 2. Load the deliberately messy dataset

Download `Messy_Employee_dataset.csv` from the [Kaggle Messy Employee Dataset](https://www.kaggle.com/datasets/desolution01/messy-employee-dataset).

The loader checks these common locations automatically:

- Kaggle Notebook: `/kaggle/input/messy-employee-dataset/Messy_Employee_dataset.csv`
- Local Jupyter/VS Code: `Messy_Employee_dataset.csv` beside this notebook
- Local data folder: `data/Messy_Employee_dataset.csv`

Column names are converted to `snake_case` immediately for consistent Python access. This changes only headers; the raw values remain untouched in the **before** snapshot.

In [ ]:
candidate_paths = [
    Path("/kaggle/input/messy-employee-dataset/Messy_Employee_dataset.csv"),
    Path("Messy_Employee_dataset.csv"),
    Path("data/Messy_Employee_dataset.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Messy_Employee_dataset.csv was not found. Download it from the Kaggle link "
        "above and place it beside this notebook or inside a data/ folder."
    )


def to_snake_case(column_name):
    name = str(column_name).strip()
    name = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return name.strip("_").lower()


raw_df = pd.read_csv(data_path)
raw_df.columns = [to_snake_case(column) for column in raw_df.columns]

before_df = raw_df.copy(deep=True)

print(f"Loaded: {data_path}")
print(f"Raw shape: {before_df.shape[0]:,} rows × {before_df.shape[1]:,} columns")
display(before_df.head())

In [ ]:
REQUIRED_COLUMNS = {
    "employee_id",
    "first_name",
    "last_name",
    "age",
    "department_region",
    "status",
    "join_date",
    "salary",
    "email",
    "phone",
    "performance_score",
    "remote_work",
}

missing_columns = REQUIRED_COLUMNS.difference(before_df.columns)

if missing_columns:
    raise ValueError(
        f"The dataset is missing required columns: {sorted(missing_columns)}. "
        f"Available columns: {before_df.columns.tolist()}"
    )

print("Required schema check passed.")

## 3. Initial data-quality report

A professional quality report should detect more than `NaN`. It should also expose:

- hidden missing markers such as `unknown`, `?`, `none`, and blank text;
- duplicate rows;
- columns stored in the wrong data type;
- domain and format anomalies, such as impossible ages, non-positive salaries, invalid emails, malformed phone numbers, invalid dates, and unexpected category values.

The `expected_dtype` column describes the analysis-ready target, while `dtype_accurate` reports whether the current pandas dtype meets that target.

In [ ]:
MISSING_MARKERS = {"", "na", "n/a", "null", "none", "unknown", "?", "-"}

BEFORE_EXPECTED_DTYPES = {
    "employee_id": "string",
    "first_name": "string",
    "last_name": "string",
    "age": "integer",
    "department_region": "string",
    "status": "category",
    "join_date": "datetime",
    "salary": "float",
    "email": "string",
    "phone": "string",
    "performance_score": "category",
    "remote_work": "boolean",
}

STATUS_VALUES = {"active", "inactive", "pending"}
PERFORMANCE_VALUES = {"poor", "average", "good", "excellent"}
BOOLEAN_VALUES = {"true", "false", "yes", "no", "y", "n", "1", "0"}
EMAIL_PATTERN = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"


def dtype_matches(series, expected):
    dtype_name = str(series.dtype)
    checks = {
        "string": dtype_name == "string",
        "integer": is_integer_dtype(series.dtype),
        "float": is_float_dtype(series.dtype),
        "datetime": is_datetime64_any_dtype(series.dtype),
        "boolean": is_bool_dtype(series.dtype),
        "category": is_categorical_dtype(series.dtype),
    }
    return bool(checks.get(expected, False))


def hidden_missing_count(series):
    if not (series.dtype == "object" or str(series.dtype).startswith("string")):
        return 0
    normalized = series.astype("string").str.strip().str.lower()
    return int(normalized.isin(MISSING_MARKERS).sum())


def parse_salary(series):
    cleaned = (
        series.astype("string")
        .str.replace(r"[^0-9.\-]", "", regex=True)
        .replace("", pd.NA)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def anomaly_mask(dataframe, column):
    series = dataframe[column]
    present = series.notna()

    if column == "employee_id":
        values = series.astype("string").str.strip()
        return present & ~values.str.match(r"^EMP-?\d+$", na=False)
    if column == "age":
        numeric = pd.to_numeric(series, errors="coerce")
        return present & (numeric.isna() | numeric.lt(18) | numeric.gt(70))
    if column == "salary":
        numeric = parse_salary(series)
        return present & (numeric.isna() | numeric.le(0))
    if column == "join_date":
        parsed = pd.to_datetime(series, format="mixed", errors="coerce")
        return present & (parsed.isna() | parsed.gt(REFERENCE_DATE))
    if column == "email":
        values = series.astype("string").str.strip()
        return present & ~values.str.match(EMAIL_PATTERN, na=False)
    if column == "phone":
        digits = series.astype("string").str.replace(r"\D", "", regex=True)
        return present & ~digits.str.len().between(10, 15)
    if column == "status":
        values = series.astype("string").str.strip().str.lower()
        return present & ~values.isin(STATUS_VALUES)
    if column == "performance_score":
        values = series.astype("string").str.strip().str.lower()
        return present & ~values.isin(PERFORMANCE_VALUES)
    if column == "remote_work":
        values = series.astype("string").str.strip().str.lower()
        return present & ~values.isin(BOOLEAN_VALUES)
    return pd.Series(False, index=dataframe.index)


def build_quality_report(dataframe, expected_dtypes):
    rows = []
    for column in dataframe.columns:
        expected = expected_dtypes.get(column, "review")
        anomaly_count = int(anomaly_mask(dataframe, column).sum()) if column in expected_dtypes else 0
        rows.append(
            {
                "column": column,
                "dtype": str(dataframe[column].dtype),
                "expected_dtype": expected,
                "dtype_accurate": dtype_matches(dataframe[column], expected),
                "explicit_nulls": int(dataframe[column].isna().sum()),
                "hidden_missing_markers": hidden_missing_count(dataframe[column]),
                "unique_values": int(dataframe[column].nunique(dropna=True)),
                "format_or_range_anomalies": anomaly_count,
            }
        )
    return pd.DataFrame(rows).set_index("column")


before_quality = build_quality_report(before_df, BEFORE_EXPECTED_DTYPES)
display(before_quality)

print(f"Exact duplicate rows: {before_df.duplicated().sum():,}")
print(
    "Columns with correct dtype: "
    f"{before_quality['dtype_accurate'].sum():,} of {len(before_quality):,}"
)

In [ ]:
# Review raw category variants before standardisation.
for column in ["status", "performance_score", "remote_work", "department_region"]:
    print(f"\n{column.upper()} — sample values")
    display(before_df[column].value_counts(dropna=False).head(12).to_frame("count"))

## 4. Missing-data decisions

Different columns require different treatment. Applying one rule to every field would damage data quality.

| Column | Strategy | Professional justification |
|---|---|---|
| `employee_id` | Delete rows with missing/invalid IDs | A primary identifier cannot be reconstructed reliably. |
| `first_name`, `last_name` | Constant value `Unknown` | Mode imputation would assign another employee's name and create false identity data. |
| `age` | Department median, then global median | Median is robust to extreme or invalid ages and retains useful rows. |
| `department`, `region`, `status` | Mode | These are low-cardinality categorical fields; mode preserves a valid category. |
| `join_date` | Median valid date | Records are not a time-ordered sequence, so forward fill could copy an unrelated employee's date. |
| `salary` | Department median, then global median | Department-aware median respects pay structure and is resistant to salary outliers. |
| `performance_score`, `remote_work` | Mode | Appropriate for categorical/boolean fields with a limited valid set. |
| `email`, `phone` | Retain as missing | Contact details are unique. Inventing or mode-filling them would create false personal data. |

**Forward fill is deliberately not used.** It is suitable when adjacent records have a meaningful sequence, which independent employee rows do not.

## 5. Clean and standardise the data

A cleaning log records how many rows or values each operation affects. This makes the workflow auditable rather than a sequence of unexplained transformations.

In [ ]:
df = before_df.copy(deep=True)
cleaning_log = []

# Remove accidental export-index columns, if present.
unnamed_columns = [column for column in df.columns if column.startswith("unnamed")]
if unnamed_columns:
    df = df.drop(columns=unnamed_columns)
    cleaning_log.append(
        {
            "issue": "Export index columns",
            "action": f"Dropped {unnamed_columns}",
            "affected": len(unnamed_columns),
            "reason": "They are file-export artifacts, not employee attributes.",
        }
    )

# Convert hidden text markers to true pandas missing values.
hidden_markers_before = 0
for column in df.select_dtypes(include=["object", "string"]).columns:
    values = df[column].astype("string").str.strip()
    marker_mask = values.str.lower().isin(MISSING_MARKERS)
    hidden_markers_before += int(marker_mask.sum())
    df[column] = values.mask(marker_mask, pd.NA)

cleaning_log.append(
    {
        "issue": "Hidden missing markers",
        "action": "Converted to pd.NA",
        "affected": hidden_markers_before,
        "reason": "All missing values should use one consistent representation.",
    }
)

# Remove exact duplicates first.
exact_duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
cleaning_log.append(
    {
        "issue": "Exact duplicate rows",
        "action": "Removed duplicates; kept first occurrence",
        "affected": exact_duplicates_removed,
        "reason": "Exact copies would double-count employees and bias analysis.",
    }
)

print(f"Hidden missing markers converted: {hidden_markers_before:,}")
print(f"Exact duplicate rows removed:      {exact_duplicates_removed:,}")

In [ ]:
# Employee ID: extract the numeric component and standardize as EMP-XXXX.
original_employee_id = df["employee_id"].copy()
employee_digits = df["employee_id"].astype("string").str.extract(r"(\d+)", expand=False)
df["employee_id"] = employee_digits.map(
    lambda value: f"EMP-{int(value):04d}" if pd.notna(value) else pd.NA
).astype("string")
employee_id_changes = int(
    original_employee_id.astype("string").fillna("<NA>").ne(
        df["employee_id"].fillna("<NA>")
    ).sum()
)

# Names: trim repeated spaces and apply title case.
for column in ["first_name", "last_name"]:
    df[column] = (
        df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )

# Split the compound department-region field using the final hyphen.
parts = df["department_region"].astype("string").str.rsplit("-", n=1, expand=True)
df["department"] = parts[0].str.strip().str.title()
df["region"] = parts[1].str.strip().str.title() if parts.shape[1] > 1 else pd.NA

department_name_fixes = {
    "Hr": "HR",
    "It": "IT",
    "Qa": "QA",
    "Devops": "DevOps",
}
df["department"] = df["department"].replace(department_name_fixes)
df = df.drop(columns="department_region")

cleaning_log.append(
    {
        "issue": "Inconsistent IDs and compound field",
        "action": "Standardized employee IDs and split department/region",
        "affected": employee_id_changes,
        "reason": "Atomic, consistently formatted fields are safer to group and join.",
    }
)

display(df[["employee_id", "first_name", "last_name", "department", "region"]].head())

In [ ]:
# Standardize categorical values.
status_map = {
    "active": "Active",
    "inactive": "Inactive",
    "pending": "Pending",
}
performance_map = {
    "poor": "Poor",
    "average": "Average",
    "good": "Good",
    "excellent": "Excellent",
}
boolean_map = {
    "true": True,
    "yes": True,
    "y": True,
    "1": True,
    "false": False,
    "no": False,
    "n": False,
    "0": False,
}

df["status"] = df["status"].astype("string").str.strip().str.lower().map(status_map)
df["performance_score"] = (
    df["performance_score"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(performance_map)
)
df["remote_work"] = (
    df["remote_work"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(boolean_map)
)

# Normalize and validate email addresses.
df["email"] = df["email"].astype("string").str.strip().str.lower()
invalid_email_mask = df["email"].notna() & ~df["email"].str.match(EMAIL_PATTERN, na=False)
invalid_emails = int(invalid_email_mask.sum())
df.loc[invalid_email_mask, "email"] = pd.NA

# Keep digits only in phone numbers; valid international lengths are 10–15 digits.
df["phone"] = df["phone"].astype("string").str.replace(r"\D", "", regex=True)
invalid_phone_mask = df["phone"].notna() & ~df["phone"].str.len().between(10, 15)
invalid_phones = int(invalid_phone_mask.sum())
df.loc[invalid_phone_mask, "phone"] = pd.NA

cleaning_log.extend(
    [
        {
            "issue": "Invalid email format",
            "action": "Converted invalid values to pd.NA",
            "affected": invalid_emails,
            "reason": "A false email is more harmful than an explicitly missing one.",
        },
        {
            "issue": "Invalid phone format",
            "action": "Removed non-digits and set invalid lengths to pd.NA",
            "affected": invalid_phones,
            "reason": "Phone numbers are identifiers, not numeric quantities.",
        },
    ]
)

print(f"Invalid emails set to missing: {invalid_emails:,}")
print(f"Invalid phones set to missing: {invalid_phones:,}")

In [ ]:
# Numeric conversion and domain rules.
df["age"] = pd.to_numeric(df["age"], errors="coerce")
invalid_age_mask = df["age"].notna() & ~df["age"].between(18, 70)
invalid_ages = int(invalid_age_mask.sum())
df.loc[invalid_age_mask, "age"] = np.nan

# Cast to float before IQR capping so non-integer fence values are valid.
df["salary"] = parse_salary(df["salary"]).astype("float64")
invalid_salary_mask = df["salary"].notna() & df["salary"].le(0)
invalid_salaries = int(invalid_salary_mask.sum())
df.loc[invalid_salary_mask, "salary"] = np.nan

# Mixed-format date parsing; impossible or future dates become missing.
original_join_date_present = df["join_date"].notna()
df["join_date"] = pd.to_datetime(df["join_date"], format="mixed", errors="coerce")
invalid_date_mask = original_join_date_present & (
    df["join_date"].isna() | df["join_date"].gt(REFERENCE_DATE)
)
invalid_dates = int(invalid_date_mask.sum())
df.loc[df["join_date"].gt(REFERENCE_DATE), "join_date"] = pd.NaT

cleaning_log.extend(
    [
        {
            "issue": "Impossible age values",
            "action": "Set ages outside 18–70 to missing",
            "affected": invalid_ages,
            "reason": "They violate the employee-age business rule.",
        },
        {
            "issue": "Invalid/non-positive salary",
            "action": "Removed formatting and set invalid values to missing",
            "affected": invalid_salaries,
            "reason": "Salary must be numeric and positive.",
        },
        {
            "issue": "Invalid or future join date",
            "action": "Parsed mixed formats and set invalid dates to NaT",
            "affected": invalid_dates,
            "reason": "Employees cannot join after the reference date.",
        },
    ]
)

print(f"Invalid ages:      {invalid_ages:,}")
print(f"Invalid salaries:  {invalid_salaries:,}")
print(f"Invalid dates:     {invalid_dates:,}")

## 6. Apply the documented missing-value strategies

Imputation is performed **after** invalid values have been converted to missing values. Otherwise, malformed entries would contaminate statistics such as the median.

In [ ]:
missing_before_imputation = df.isna().sum().rename("before_imputation")

# A missing primary key cannot be recovered safely.
missing_id_rows = int(df["employee_id"].isna().sum())
df = df.dropna(subset=["employee_id"]).copy()

# Names: use an explicit placeholder rather than assigning another person's name.
df["first_name"] = df["first_name"].fillna("Unknown")
df["last_name"] = df["last_name"].fillna("Unknown")


def safe_mode(series, fallback="Unknown"):
    modes = series.dropna().mode()
    return modes.iloc[0] if not modes.empty else fallback


# Low-cardinality categories: mode.
for column in ["department", "region", "status", "performance_score", "remote_work"]:
    df[column] = df[column].fillna(safe_mode(df[column]))

# Numeric fields: department-aware median, then global median.
age_group_median = df.groupby("department", observed=True)["age"].transform("median")
salary_group_median = df.groupby("department", observed=True)["salary"].transform("median")

df["age"] = df["age"].fillna(age_group_median).fillna(df["age"].median())
df["salary"] = (
    df["salary"]
    .fillna(salary_group_median)
    .fillna(df["salary"].median())
)

# Date: dataset-wide median valid date; forward fill is inappropriate for independent rows.
median_join_date = df["join_date"].median()
df["join_date"] = df["join_date"].fillna(median_join_date)

missing_after_imputation = df.isna().sum().rename("after_imputation")
imputation_summary = pd.concat(
    [missing_before_imputation, missing_after_imputation],
    axis=1,
).fillna(0).astype(int)
imputation_summary["values_resolved"] = (
    imputation_summary["before_imputation"] - imputation_summary["after_imputation"]
)

cleaning_log.append(
    {
        "issue": "Missing primary key",
        "action": "Deleted affected rows",
        "affected": missing_id_rows,
        "reason": "Employee identity cannot be inferred safely.",
    }
)

display(imputation_summary)

## 7. IQR outlier detection and treatment

The **Interquartile Range (IQR)** is resistant to extreme values:

\[
IQR = Q_3 - Q_1
\]

Values below \(Q_1 - 1.5(IQR)\) or above \(Q_3 + 1.5(IQR)\) are flagged.

Decisions:

- **Age:** retain IQR outliers after applying the valid 18–70 business range. An uncommon but valid employee age is plausible and should not be altered merely for being rare.
- **Salary:** cap IQR outliers at the lower/upper IQR fences. This preserves employees while reducing the impact of synthetic extreme salaries on averages and models.

In [ ]:
def iqr_details(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = series.lt(lower) | series.gt(upper)
    return q1, q3, iqr, lower, upper, mask


outlier_rows = []

age_q1, age_q3, age_iqr, age_lower, age_upper, age_outlier_mask = iqr_details(df["age"])
outlier_rows.append(
    {
        "column": "age",
        "Q1": age_q1,
        "Q3": age_q3,
        "IQR": age_iqr,
        "lower_fence": age_lower,
        "upper_fence": age_upper,
        "outliers_before": int(age_outlier_mask.sum()),
        "decision": "Retain values within the valid 18–70 business range",
    }
)

salary_q1, salary_q3, salary_iqr, salary_lower, salary_upper, salary_outlier_mask = iqr_details(df["salary"])
salary_outliers_before = int(salary_outlier_mask.sum())
df["salary"] = df["salary"].clip(lower=max(0, salary_lower), upper=salary_upper)
_, _, _, _, _, salary_outlier_mask_after = iqr_details(df["salary"])

outlier_rows.append(
    {
        "column": "salary",
        "Q1": salary_q1,
        "Q3": salary_q3,
        "IQR": salary_iqr,
        "lower_fence": salary_lower,
        "upper_fence": salary_upper,
        "outliers_before": salary_outliers_before,
        "decision": "Cap to IQR fences; retain all rows",
    }
)

outlier_summary = pd.DataFrame(outlier_rows).set_index("column")
display(outlier_summary)

cleaning_log.append(
    {
        "issue": "Salary IQR outliers",
        "action": "Capped at the IQR fences",
        "affected": salary_outliers_before,
        "reason": "Winsorisation limits influence while preserving employee records.",
    }
)

In [ ]:
# Visual validation of salary treatment.
raw_salary_numeric = parse_salary(before_df["salary"])

salary_plot_data = pd.concat(
    [
        pd.DataFrame({"version": "Before", "salary": raw_salary_numeric}),
        pd.DataFrame({"version": "After", "salary": df["salary"]}),
    ],
    ignore_index=True,
)

plt.figure(figsize=(11, 5))
sns.boxplot(data=salary_plot_data, x="version", y="salary", palette=["#E76F51", "#2A9D8F"])
plt.title("Salary Distribution Before and After Outlier Treatment")
plt.xlabel("")
plt.ylabel("Salary")
plt.tight_layout()
plt.show()

## 8. Remove business-key duplicates

Exact copies were removed earlier. Standardisation can reveal additional duplicates that were hidden by inconsistent ID formatting or text casing.

- For repeated `employee_id`, keep the most complete record.
- For records sharing first name, last name, non-missing email, and join date, keep the first most-complete record.

Requiring a non-missing email in the second rule prevents accidental deletion of different employees who happen to share a common name.

In [ ]:
df["_completeness"] = df.notna().sum(axis=1)
df = df.sort_values("_completeness", ascending=False)

duplicate_employee_ids = int(df.duplicated(subset=["employee_id"], keep="first").sum())
df = df.drop_duplicates(subset=["employee_id"], keep="first")

person_key = ["first_name", "last_name", "email", "join_date"]
duplicate_person_mask = (
    df["email"].notna()
    & df.duplicated(subset=person_key, keep="first")
)
duplicate_people = int(duplicate_person_mask.sum())
df = df.loc[~duplicate_person_mask].drop(columns="_completeness").reset_index(drop=True)

cleaning_log.extend(
    [
        {
            "issue": "Duplicate standardized employee ID",
            "action": "Kept the most complete record",
            "affected": duplicate_employee_ids,
            "reason": "An employee ID must uniquely identify one record.",
        },
        {
            "issue": "Duplicate person business key",
            "action": "Removed repeated name-email-date records",
            "affected": duplicate_people,
            "reason": "The combined key indicates the same employee despite a different raw record.",
        },
    ]
)

print(f"Duplicate employee IDs removed: {duplicate_employee_ids:,}")
print(f"Duplicate person records removed: {duplicate_people:,}")
print(f"Rows remaining: {len(df):,}")

## 9. Correct data types

IDs and phone numbers are stored as strings because arithmetic on them is meaningless. Nullable pandas dtypes (`string`, `Int64`, and `boolean`) preserve missing values without falling back to generic `object`.

In [ ]:
# Nullable string fields.
string_columns = ["employee_id", "first_name", "last_name", "email", "phone"]
for column in string_columns:
    df[column] = df[column].astype("string")

# Numeric, date, boolean, and category fields.
df["age"] = df["age"].round().astype("Int64")
df["salary"] = df["salary"].astype("float64")
df["join_date"] = pd.to_datetime(df["join_date"]).dt.normalize()
df["remote_work"] = df["remote_work"].astype("boolean")

for column in ["department", "region", "status", "performance_score"]:
    df[column] = df[column].astype("category")

CLEAN_COLUMN_ORDER = [
    "employee_id",
    "first_name",
    "last_name",
    "age",
    "department",
    "region",
    "status",
    "join_date",
    "salary",
    "email",
    "phone",
    "performance_score",
    "remote_work",
]
df = df[CLEAN_COLUMN_ORDER].sort_values("employee_id").reset_index(drop=True)

df.info()

## 10. Validate the cleaned dataset

Cleaning is not complete until the output is tested. These assertions act as lightweight data-quality tests and stop the notebook if a critical rule fails.

In [ ]:
AFTER_EXPECTED_DTYPES = {
    "employee_id": "string",
    "first_name": "string",
    "last_name": "string",
    "age": "integer",
    "department": "category",
    "region": "category",
    "status": "category",
    "join_date": "datetime",
    "salary": "float",
    "email": "string",
    "phone": "string",
    "performance_score": "category",
    "remote_work": "boolean",
}

validation_results = {
    "Employee ID has no missing values": df["employee_id"].notna().all(),
    "Employee ID is unique": df["employee_id"].is_unique,
    "No exact duplicate rows": not df.duplicated().any(),
    "Age is within 18–70": df["age"].between(18, 70).all(),
    "Salary is positive": df["salary"].gt(0).all(),
    "Join date is not in the future": df["join_date"].le(REFERENCE_DATE).all(),
    "Status values are valid": set(df["status"].dropna()).issubset({"Active", "Inactive", "Pending"}),
    "Performance values are valid": set(df["performance_score"].dropna()).issubset({"Poor", "Average", "Good", "Excellent"}),
    "All target dtypes are correct": all(
        dtype_matches(df[column], expected)
        for column, expected in AFTER_EXPECTED_DTYPES.items()
    ),
}

validation_table = (
    pd.Series(validation_results, name="passed")
    .rename_axis("quality_test")
    .to_frame()
)
display(validation_table)

assert validation_table["passed"].all(), "One or more data-quality tests failed."
print("All critical data-quality tests passed.")

## 11. Before-versus-after summary

`effective_missing_values` combines true nulls with hidden text markers in the raw dataset. This prevents the initial quality estimate from looking artificially better simply because values such as `unknown` were stored as text.

Email and phone nulls may remain by design: manufacturing contact information would be less accurate than explicitly recording it as missing.

In [ ]:
after_quality = build_quality_report(df, AFTER_EXPECTED_DTYPES)

before_dtype_accuracy = (
    before_quality.loc[before_quality["expected_dtype"].ne("review"), "dtype_accurate"].mean()
)
after_dtype_accuracy = after_quality["dtype_accurate"].mean()

before_effective_missing = int(
    before_quality["explicit_nulls"].sum()
    + before_quality["hidden_missing_markers"].sum()
)
after_effective_missing = int(
    after_quality["explicit_nulls"].sum()
    + after_quality["hidden_missing_markers"].sum()
)

before_after_summary = pd.DataFrame(
    {
        "metric": [
            "Row count",
            "Explicit null count",
            "Effective missing values",
            "Exact duplicate rows",
            "Format/range anomalies",
            "Dtype accuracy",
        ],
        "before": [
            len(before_df),
            int(before_df.isna().sum().sum()),
            before_effective_missing,
            int(before_df.duplicated().sum()),
            int(before_quality["format_or_range_anomalies"].sum()),
            before_dtype_accuracy,
        ],
        "after": [
            len(df),
            int(df.isna().sum().sum()),
            after_effective_missing,
            int(df.duplicated().sum()),
            int(after_quality["format_or_range_anomalies"].sum()),
            after_dtype_accuracy,
        ],
    }
).set_index("metric")

formatted_summary = before_after_summary.copy().astype("object")
formatted_summary.loc["Dtype accuracy", "before"] = f"{before_dtype_accuracy:.1%}"
formatted_summary.loc["Dtype accuracy", "after"] = f"{after_dtype_accuracy:.1%}"

for metric in formatted_summary.index.difference(["Dtype accuracy"]):
    formatted_summary.loc[metric, "before"] = f"{before_after_summary.loc[metric, 'before']:,.0f}"
    formatted_summary.loc[metric, "after"] = f"{before_after_summary.loc[metric, 'after']:,.0f}"

display(formatted_summary)

In [ ]:
# Per-column null comparison for columns present in either version.
null_comparison = pd.concat(
    [
        before_df.isna().sum().rename("before_explicit_nulls"),
        df.isna().sum().rename("after_explicit_nulls"),
    ],
    axis=1,
).fillna(0).astype(int)

display(null_comparison)

total_missing_plot = pd.DataFrame(
    {
        "version": ["Before", "After"],
        "effective_missing_values": [before_effective_missing, after_effective_missing],
    }
)
plt.figure(figsize=(8, 5))
ax = sns.barplot(
    data=total_missing_plot,
    x="version",
    y="effective_missing_values",
    hue="version",
    palette={"Before": "#E76F51", "After": "#2A9D8F"},
    legend=False,
)
ax.set_title("Effective Missing Values Before and After Cleaning")
ax.set_xlabel("")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}", padding=4)
plt.tight_layout()
plt.show()

In [ ]:
cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)

## 12. Save the cleaned dataset

Dates are written in ISO `YYYY-MM-DD` format, which is unambiguous and portable. CSV files do not store pandas dtype metadata, so IDs and phone numbers should be read as strings in future workflows.

In [ ]:
df.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d")

assert OUTPUT_PATH.exists(), "Cleaned CSV export failed."

# Round-trip verification: confirm the exported file is readable and has the expected shape.
exported_check = pd.read_csv(
    OUTPUT_PATH,
    dtype={"employee_id": "string", "phone": "string"},
)
assert exported_check.shape == df.shape
assert exported_check.columns.tolist() == df.columns.tolist()

print(f"Saved cleaned dataset: {OUTPUT_PATH.resolve()}")
print(f"Exported shape: {exported_check.shape[0]:,} rows × {exported_check.shape[1]:,} columns")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024:,.1f} KB")

## 13. Conclusion

The raw employee data has been converted into a structured, validated, and analysis-ready dataset. The workflow:

- distinguished explicit nulls from hidden missing markers;
- documented a column-specific missing-data strategy;
- standardized IDs, text, categories, contact fields, and mixed dates;
- converted invalid domain values to missing before imputation;
- used department-aware median imputation for age and salary;
- detected numeric outliers with the IQR method and treated salary outliers without deleting records;
- removed exact and business-key duplicates;
- applied correct nullable pandas data types;
- tested critical business rules with assertions; and
- exported a new cleaned CSV.

The cleaned file can now support HR reporting, employee segmentation, salary analysis, dashboarding, and downstream machine-learning work. The remaining missing email or phone values are intentional: transparent missingness is preferable to invented personal data.

## References

- [Kaggle — Messy Employee Dataset](https://www.kaggle.com/datasets/desolution01/messy-employee-dataset)
- [pandas — Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas — `DataFrame.astype`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)
- [pandas — Time series and date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [YouTube — Data Cleaning with Python Pandas: Hands-On Tutorial](https://www.youtube.com/watch?v=I7DZP4rVQOU)

---

**Checklist completed:** quality report, missing-value decisions, duplicate removal, standardisation, IQR outlier handling, dtype correction, before/after comparison, validation, and cleaned CSV export.